# 07 · Market reaction after each filing

**Reads** `stg_clean_10Ks` (dbt) · **Writes** `fact_market_reaction`

For every filing: 1-, 5- and 20-trading-day return from the first close on or after the filing date, and the
annualised volatility of the following 20 trading days. Returns are rounded to 4 decimals (the old 2-decimal
rounding turned a 1.4% move into 1%). Logic and edge cases: `trading_sentiment/market.py`.

In [ ]:
%pip install yfinance

In [ ]:
%restart_python

In [ ]:
# Make the repository's Python package importable (Databricks Git folder: cwd is this notebook's folder)
import os, sys
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from trading_sentiment import table

In [ ]:
import pandas as pd
import yfinance as yf
from datetime import timedelta
from pyspark.sql import functions as F
from pyspark.sql.functions import pandas_udf
from pyspark.sql.types import DoubleType, StructField, StructType

import trading_sentiment, trading_sentiment.market as market_module
from trading_sentiment.market import NO_REACTION, extract_close, reaction_from_prices

# The pandas UDF below runs on the executors, which cannot import this repository's package:
# ship the modules inside the pickled function instead.
from pyspark import cloudpickle
for module in (trading_sentiment, market_module):
    cloudpickle.register_pickle_by_value(module)

def get_market_reaction(ticker, filing_date):
    try:
        start = pd.Timestamp(filing_date)
        prices = extract_close(yf.download(ticker, start=start, end=start + timedelta(days=40),
                                           progress=False, auto_adjust=True))
        return reaction_from_prices(prices, start)
    except Exception as exc:                      # was a bare `except:` that hid every failure
        print(f"{ticker} {filing_date}: {exc}")
        return dict(NO_REACTION)

schema = StructType([StructField(name, DoubleType()) for name in ("return_1d", "return_5d", "return_20d", "volatility_20d")])

@pandas_udf(schema)
def market_udf(tickers: pd.Series, dates: pd.Series) -> pd.DataFrame:
    return pd.DataFrame([get_market_reaction(t, d) for t, d in zip(tickers, dates)])

In [ ]:
filings = (spark.table(table("stg_clean_10Ks"))
                .select("cik", "tickers", "filing_date", "market_cap").distinct().orderBy(F.desc("market_cap")))

market = (filings.withColumn("m", market_udf("tickers", "filing_date"))
    .select("cik", "tickers", "filing_date",
            *[F.round(f"m.{c}", 4).alias(c) for c in ("return_1d", "return_5d", "return_20d", "volatility_20d")]))
market.write.format("delta").mode("overwrite").saveAsTable(table("fact_market_reaction"))
display(market.limit(20))

Prices are adjusted closes from Yahoo Finance; filings published after the close are measured from that day's close,
so `return_1d` can include part of the first reaction. Treat these as descriptive, not as a tradable signal.